# 01a · 用 vLLM 測量 SFT 模型的錯誤率

**目的**：讓 SFT 模型回答 train 裡約 500 題，跟標準答案比對，回答兩個問題：

1. 模型自己會犯多少錯？→ 決定 DPO 配對要靠「模型自己的錯」還是「規則造錯」
2. 錯在哪裡？（哪個題型、哪種錯誤）→ 決定規則造錯要加重哪幾類

**Kaggle 設定**
- Accelerator：**GPU T4 x2**（P100 不支援 vLLM，不要選）
- Internet：**On**（要下載 vLLM 和模型）
- Secrets：`HF_TOKEN`（跟 00 一樣）

**執行方式：背景模式**

右上角 **Save Version → Save & Run All (Commit) → Save**。Kaggle 會開一台新機器從頭跑到尾，你可以關掉瀏覽器。
跑完後到 notebook 頁面 → 右上 **Version** → 選這次的 version → **Output** 頁籤下載檔案；**Logs** 頁籤可以看完整的執行紀錄。

**預估時間**：15～25 分鐘（不含排隊）。

**如果 vLLM 載入失敗**：程式會在第 9 格直接停下（約 10 分鐘內），**不會**改用 transformers 慢慢跑。
到 **Logs** 頁籤搜尋 `VLLM_LOAD_FAILED`，把下面的錯誤訊息傳給 Claude。

## 1. 安裝 vLLM

**重要：這一格必須是整本 notebook 第一個執行的格子。**

`pip install vllm` 會連帶安裝它指定版本的 PyTorch。如果在這之前已經 `import torch`，記憶體裡就會是舊版 torch，接下來會出現各種奇怪的錯誤。如果真的遇到，請 **Run → Restart & clear cell outputs**，再從頭跑一次。

| 寫法 | 意思 |
|---|---|
| `-q` / `-U` | 只印錯誤 / 升級到指定版本 |
| `vllm==0.34.0` | **鎖定版本**。上一次跑的時候確認了 0.34.0 在 T4 上可以載入模型。沒有鎖版本的話，下週 vLLM 發新版，同一本 notebook 可能就跑不動了 |
| `nvidia-cutlass-dsl` | 補上 vLLM 0.34.0 漏掉的套件。它在載入的最後一步（kernel warmup）會 `import cutlass`（原本是給 MiniMax-M3 模型用的），但安裝 vLLM 時不會自動裝這個套件 |
| `datasets` | 跟 vLLM 寫在同一行一起裝，pip 才會挑出跟新版 `huggingface-hub` 相容的版本 |

最後會出現 `ERROR: pip's dependency resolver...` 和一長串衝突清單。這是**警告**，代表 Kaggle 預裝的其他套件（gradio、cudf 等）版本對不上。我們沒用到那些套件，可以忽略。只要清單裡沒有 `datasets`、`transformers`、`torch`、`vllm` 就沒問題。

安裝完會**直接在這個 notebook 裡** `import cutlass`。如果這裡就失敗，整本會在 5 分鐘內停下，不用等到載入模型才發現。

> **踩過的坑：檢查要測在「真正會用到的地方」**
> 上一版用 `subprocess` 開新的 Python 程序去 import，結果印出 `cutlass OK`，vLLM 卻還是找不到 cutlass。
> 因為 `nvidia-cutlass-dsl` 是用 **`.pth` 檔**把自己加進搜尋路徑，而 Python **只在啟動時**讀 `.pth`。
> 新開的程序有讀到，所以找得到；notebook 本身的程序在安裝前就啟動了，所以找不到。
> 修正方法是用 `site.addsitedir()` 讓 notebook 重新讀一次 `.pth`，再直接在 notebook 裡 import 檢查。

In [ ]:
!pip install -q -U "vllm==0.34.0" nvidia-cutlass-dsl datasets

# nvidia-cutlass-dsl 是用 .pth 檔把自己加進 Python 搜尋路徑，但 .pth 只在 Python 啟動時讀取。
# 這個 notebook 的 Python 在安裝前就啟動了，所以要手動讓它重新讀一次。
import site, importlib
for d in site.getsitepackages():
    site.addsitedir(d)          # 重新處理這個資料夾裡的 .pth 檔，把新路徑加進 sys.path
importlib.invalidate_caches()   # 清掉 import 的快取，讓 Python 重新掃描資料夾

# 保險：問一個新的 Python 程序「cutlass 裝在哪」，把那個路徑也加進來（不管它用什麼方式安裝都有效）
import subprocess, sys
where = subprocess.run([sys.executable, "-c", "import cutlass, os; print(os.path.dirname(list(cutlass.__path__)[0]))"],
                       capture_output=True, text=True, check=True).stdout.strip()
if where not in sys.path:
    sys.path.insert(0, where)
    importlib.invalidate_caches()
print("cutlass 所在路徑:", where)

# 直接在「這個」程序裡 import —— vLLM 等一下就是在這個程序裡跑，要測就要測同一個地方
import cutlass
print("cutlass OK（notebook 本身找得到）:", list(cutlass.__path__)[0])

## 2. 記錄版本

vLLM 大約每一到兩週就發布新版。**實驗紀錄一定要寫下版本號**，之後重跑結果不一樣時，才能知道是不是版本造成的。這些版本號會一起存進結果檔。

In [ ]:
import importlib.metadata as md_
VERSIONS = {p: md_.version(p) for p in ["vllm", "torch", "transformers", "datasets", "huggingface-hub", "nvidia-cutlass-dsl"]}
print(VERSIONS)

## 3. 登入 Hugging Face

跟 00 一樣用 Kaggle Secrets。這次另外把 token 放進環境變數 `HF_TOKEN`，因為 vLLM 下載模型時會讀這個變數。

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
login(token=os.environ["HF_TOKEN"])

## 4. 實驗參數（全部集中在這裡）

| 參數 | 值 | 意思 | 什麼時候要改 |
|---|---|---|---|
| `MODEL_ID` | merged-v1 | 要測的模型 | 之後測 DPO 模型時換掉 |
| `N_TARGET` | 500 | 大約要抽幾題 | 想快速試跑可以改 200 |
| `MIN_PER_CATEGORY` | 40 | 每個題型至少抽幾題 | off_topic 只有 105 題，按比例只會抽到 13 題，太少就看不出錯誤率 |
| `N_SAMPLES` | 4 | 抽樣模式下每題回答幾次 | 越多越容易抓到錯誤，但時間也成正比增加 |
| `TEMPERATURE` | 0.8 | 抽樣的隨機程度 | 0 = 每次都選機率最高的字；越高越敢冒險 |
| `TOP_P` | 0.95 | 只從累積機率前 95% 的字裡面挑 | 擋掉機率極低的怪字，避免產生亂碼 |
| `MAX_TOKENS` | 512 | 每個回答最多幾個 token | 最長的 multi 題大約 300 token，512 很安全 |
| `SEED` | 42 | 隨機種子 | 固定之後，抽題和抽樣都可以重現 |
| `SMOKE_N` | 20 | 冒煙測試題數 | — |

In [ ]:
MODEL_ID   = "yoyu001/grid-dispatch-A-instruct-merged-v1"
DATASET_ID = "yoyu001/grid-dispatch-notes-zh"
SPLIT      = "train"          # 只用 train，絕對不碰 test

N_TARGET         = 500
MIN_PER_CATEGORY = 40
N_SAMPLES        = 4
TEMPERATURE      = 0.8
TOP_P            = 0.95
MAX_TOKENS       = 512
SEED             = 42
SMOKE_N          = 20

OUT_DIR = "/kaggle/working"   # 直接放根目錄：Output 頁籤和下載連結都比較不會出問題

## 5. 檢查 GPU

vLLM 要求 compute capability ≥ 7.0，T4 是 **7.5**，符合。但 T4 有兩個限制：

- **不支援 bfloat16** → 等一下要指定 `dtype="half"`（float16）
- **不支援 FlashAttention-2** → vLLM 會自動改用其他 attention 實作（例如 Triton），載入時的 log 會寫是哪一個，可以看一下

> **為什麼這裡用 `nvidia-smi`，不用 `torch.cuda`？**
> 只要呼叫 `torch.cuda.xxx`，CUDA 就會在這個 Python 程序裡初始化。vLLM 預設會另外開子程序來跑模型，
> 如果父程序已經初始化過 CUDA，子程序常常會出錯（`Cannot re-initialize CUDA in forked subprocess`）。
> `nvidia-smi` 是外部指令，不會碰到 Python 裡的 CUDA，所以比較安全。上一版在這裡先動了 `torch.cuda`，這是 vLLM 失敗的可能原因之一。

In [ ]:
import subprocess
GPU_INFO = subprocess.run(["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()
print(GPU_INFO)
assert GPU_INFO, "沒有 GPU！請到 Settings → Accelerator 選 GPU T4 x2"

## 6. 分層抽題（stratified sampling）

如果直接隨機抽 500 題，大題型（multi 1085 題）會抽很多，小題型（off_topic 105 題）可能只抽到十幾題，錯誤率會很不準。

**分層抽樣**的作法是每個題型各抽各的：先按比例分配題數，再保證每個題型至少有 `MIN_PER_CATEGORY` 題。

In [ ]:
import random, json, collections
from datasets import load_dataset

ds  = load_dataset(DATASET_ID, split=SPLIT)
rng = random.Random(SEED)

by_cat = collections.defaultdict(list)
for i, c in enumerate(ds["category"]):
    by_cat[c].append(i)

plan, picked = {}, []
for c, idxs in sorted(by_cat.items()):
    k = max(MIN_PER_CATEGORY, round(N_TARGET * len(idxs) / len(ds)))
    k = min(k, len(idxs))
    plan[c] = k
    picked += rng.sample(idxs, k)
rng.shuffle(picked)            # 打散順序，讓每一批都混到各種題型

probe = ds.select(picked)
print("每類抽題數:", plan, "→ 共", len(probe), "題")

## 7. 準備 prompt 和標準答案

資料裡的 `messages` 有 system / user / assistant 三段：

- **prompt** = system + user（拿掉 assistant，讓模型自己回答）
- **gold** = assistant 的內容（標準答案）

In [ ]:
def to_prompt(row):
    return [m for m in row["messages"] if m["role"] != "assistant"]

def gold_of(row):
    return next(m["content"] for m in row["messages"] if m["role"] == "assistant")

prompts = [to_prompt(r) for r in probe]
golds   = [gold_of(r) for r in probe]
meta    = [{"id": r["id"], "category": r["category"]} for r in probe]
print(json.dumps(prompts[0], ensure_ascii=False, indent=2)[:800])

## 8. 評分函式

規則跟 SFT 評估一致：**比對 `intent` 和 `constraints`**。

- constraints 不管順序
- `explanation` 不評分，因為同一個意思可以有很多種說法
- 每個錯誤都會貼上標籤，01b 會依照這些標籤決定要造哪一類的錯：

| 標籤 | 意思 | 危險程度 |
|---|---|---|
| `missed_refuse` | 該拒絕卻照做了 | 🔴 最危險 |
| `false_refuse` | 不該拒絕卻拒絕 | 🟡 |
| `wrong_intent` | intent 判斷錯誤（例如 no_action 判成 set_constraints） | 🟡 |
| `missing_constraint` / `extra_constraint` | 漏掉一條限制 / 多出一條限制 | 🟠 |
| `wrong_value` / `wrong_valid_until` / `wrong_action` | 限制的數值、時效或動作寫錯 | 🟠 |
| `json_invalid` / `schema_invalid` | 輸出不是合法的 JSON，或 JSON 結構不對 | 🟠 系統直接無法使用 |

這個函式已經在本機用你資料集的真實樣本測試過了。

In [ ]:
import json

def parse(text):
    """Turn model text into a dict, or None if it is not valid JSON."""
    t = text.strip()
    if t.startswith("```"):                      # strip ```json ... ``` fences
        t = t.strip("`")
        t = t[t.find("{"):] if "{" in t else t
    try:
        obj = json.loads(t)
    except Exception:
        s, e = t.find("{"), t.rfind("}")
        if s == -1 or e <= s:
            return None
        try:
            obj = json.loads(t[s:e + 1])
        except Exception:
            return None
    return obj if isinstance(obj, dict) else None

def _key(c):
    return (c.get("workload_id"), c.get("field"))

def _val(c):
    return {"value": json.dumps(c.get("value"), ensure_ascii=False),
            "valid_until": c.get("valid_until"),
            "action": c.get("action")}

def grade(pred_text, gold_text):
    """Compare intent + constraints (order-insensitive). explanation is not graded.

    Returns {"correct": bool, "errors": [tags], "exact_str": bool}.
    """
    gold = json.loads(gold_text)
    out = {"exact_str": pred_text.strip() == gold_text.strip()}
    pred = parse(pred_text)
    if pred is None:
        return {**out, "correct": False, "errors": ["json_invalid"]}

    g_int, p_int = gold.get("intent"), pred.get("intent")
    if p_int != g_int:
        if g_int == "refuse":
            tag = "missed_refuse"        # 危險：該拒絕卻照做
        elif p_int == "refuse":
            tag = "false_refuse"         # 不該拒絕卻拒絕
        else:
            tag = "wrong_intent"
        return {**out, "correct": False, "errors": [tag]}

    p_list = pred.get("constraints", [])
    if not isinstance(p_list, list) or any(not isinstance(c, dict) for c in p_list):
        return {**out, "correct": False, "errors": ["schema_invalid"]}

    g = {_key(c): _val(c) for c in gold.get("constraints", [])}
    p = {_key(c): _val(c) for c in p_list}
    errors = []
    if len(p) != len(p_list):
        errors.append("duplicate_constraint")
    if any(k not in p for k in g):
        errors.append("missing_constraint")
    if any(k not in g for k in p):
        errors.append("extra_constraint")
    for k in g:
        if k in p and p[k] != g[k]:
            for part in ("value", "valid_until", "action"):
                if p[k][part] != g[k][part]:
                    errors.append(f"wrong_{part}")
    errors = list(dict.fromkeys(errors))   # dedupe, keep order
    return {**out, "correct": not errors, "errors": errors}

# 自我檢查：標準答案跟自己比，一定要全對
assert all(grade(g, g)["correct"] for g in golds[:50])
print("grader OK")

## 9. 載入模型到 vLLM

### vLLM 為什麼比較快？兩個核心概念

**① PagedAttention**：模型每產生一個字，都要把前面所有字的中間結果（KV cache）留在 GPU 記憶體裡。傳統做法會替每個請求預留「最長可能長度」的空間，浪費很多。vLLM 改成像作業系統管理記憶體一樣，把 KV cache 切成小頁，用多少分多少，同一張 GPU 就能同時處理更多請求。

**② Continuous batching（連續批次）**：傳統的批次處理要等整批裡最長的那個回答寫完，才能開始下一批。vLLM 是每產生一個 token 就重新排程：有人寫完就立刻換新的請求進來，GPU 幾乎不會閒著。

### 參數說明

| 參數 | 值 | 意思 |
|---|---|---|
| `model` | merged-v1 | HF 上的模型 ID，vLLM 會自動下載 |
| `dtype` | `"half"` | 用 float16。T4 不支援 bf16，不指定的話可能會報錯 |
| `max_model_len` | 2048 | prompt + 回答的最大長度。設得越小，KV cache 能塞越多請求。你的 prompt 大約 400 token，加上回答 512 就足夠 |
| `gpu_memory_utilization` | 0.85 | vLLM 最多可以用 85% 的 GPU 記憶體。扣掉權重（約 8 GB）之後，剩下的都拿來放 KV cache。上一版用 0.90，這次留多一點空間比較保險 |
| `tensor_parallel_size` | 1 | 用幾張 GPU 一起跑同一個模型。4B 模型一張 T4 放得下，先用 1，比較單純 |
| `enforce_eager` | `True` | **關掉 CUDA graph 和 torch.compile**。這兩個是 vLLM 的加速功能，但在 T4 這種舊 GPU 上最容易編譯失敗。關掉之後大約慢 10～30%，換來穩定，仍然比 transformers 快很多 |
| `seed` | 42 | 讓抽樣結果可以重現 |

### 兩個環境變數（要在 `import vllm` 之前設定）

| 變數 | 值 | 意思 |
|---|---|---|
| `VLLM_ENABLE_V1_MULTIPROCESSING` | `"0"` | vLLM 預設會另外開一個子程序來跑模型。在 notebook 裡，子程序常常因為 fork/spawn 的問題啟動失敗。設成 0 就讓模型直接在 notebook 這個程序裡跑，少一個出錯的地方 |
| `TOKENIZERS_PARALLELISM` | `"false"` | 關掉 tokenizer 的多執行緒，避免它跟上面的設定互相干擾、跳出警告 |

> 載入時請看 log 裡的兩行：**attention backend**（T4 用的是哪一種 attention 實作），以及 **KV cache 大小 / 最大並行數**（這張 GPU 可以同時處理多少請求）。

### 失敗就停下來（fail fast）

上一版失敗時會自動改用 transformers，結果在背景默默跑了一個多小時。這版改成**失敗就直接報錯停止**，同時把錯誤寫進 `01a_vllm_error.txt`，在 Logs 也能用 `VLLM_LOAD_FAILED` 搜尋到。
這是工程上的取捨：**寧可早點失敗、看得到原因，也不要默默變慢。**

In [ ]:
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import time, traceback
from vllm import LLM, SamplingParams

t0 = time.time()
try:
    llm = LLM(
        model=MODEL_ID,
        dtype="half",
        max_model_len=2048,
        gpu_memory_utilization=0.85,
        tensor_parallel_size=1,
        enforce_eager=True,
        seed=SEED,
    )
except Exception:
    err = traceback.format_exc()
    with open(f"{OUT_DIR}/01a_vllm_error.txt", "w") as f:
        f.write(f"{VERSIONS}\n{GPU_INFO}\n\n{err}")
    print("VLLM_LOAD_FAILED\n" + err[-3000:])
    raise
LOAD_SECONDS = round(time.time() - t0)
print(f"✅ vLLM 載入完成，用時 {LOAD_SECONDS} 秒")

## 10. 產生函式

把「呼叫 vLLM、整理輸出、計時」包成一個函式 `generate(prompts, n, temperature)`，後面的 greedy、抽樣、冒煙測試都用它。

### `SamplingParams` 參數

| 參數 | Greedy | 抽樣 | 意思 |
|---|---|---|---|
| `n` | 1 | 4 | 每題回答幾次 |
| `temperature` | 0 | 0.8 | 0 = 永遠選機率最高的字 |
| `top_p` | 1.0 | 0.95 | 見參數表；greedy 時不起作用 |
| `max_tokens` | 512 | 512 | 回答長度上限 |
| `seed` | — | 42 | 可以重現 |

`llm.chat()` 會自動套用模型的 chat template（把 system/user 包成 Qwen 的對話格式），跟 SFT 時的格式一致。
一次把所有 prompt 丟給 vLLM，它會自己排程、湊批次，不需要我們手動切 batch。

回傳值裡的 `finish_reason` 很重要：如果是 `"length"`，代表回答寫到 `max_tokens` 被切斷了。你之前跑 GSM8K 時，就有 34% 的回答是這樣被截斷的。

In [ ]:
def generate(prompts, n=1, temperature=0.0):
    """回傳 (texts, finish_reasons, n_output_tokens, seconds)；texts[i] 是第 i 題的 n 個回答"""
    t0 = time.time()
    sp = SamplingParams(n=n, temperature=temperature, top_p=TOP_P if temperature > 0 else 1.0,
                        max_tokens=MAX_TOKENS, seed=SEED if temperature > 0 else None)
    outs = llm.chat(prompts, sp, use_tqdm=True)
    texts   = [[o.text for o in out.outputs] for out in outs]
    reasons = [[o.finish_reason for o in out.outputs] for out in outs]
    ntok    = sum(len(o.token_ids) for out in outs for o in out.outputs)
    return texts, reasons, ntok, time.time() - t0

## 11. 冒煙測試（smoke test）：先跑 20 題

在跑完整批之前，先確認三件事：

1. 程式能跑完
2. 輸出是合法的 JSON
3. 正確率合理（SFT 模型在 train 上應該接近 100%）

如果正確率低於 50%，通常是 chat template 或模型 ID 出了問題。這時程式會停下來，不會白白浪費 GPU 時間。

In [ ]:
texts, reasons, ntok, dt = generate(prompts[:SMOKE_N], n=1, temperature=0.0)
smoke = [grade(t[0], g) for t, g in zip(texts, golds[:SMOKE_N])]
acc = sum(r["correct"] for r in smoke) / SMOKE_N
print(f"冒煙測試正確率 {acc:.0%}　|　{ntok} tokens / {dt:.1f}s = {ntok/dt:.0f} tok/s")
for i in range(2):
    print(f"\n--- {meta[i]['id']} ({meta[i]['category']}) ---\nPRED: {texts[i][0]}\nGOLD: {golds[i]}\n→ {smoke[i]}")
assert acc >= 0.5, "正確率太低，先停下來檢查 MODEL_ID / chat template，不要往下跑"

## 12. Greedy：每題回答一次

Greedy 代表模型「最有把握」的答案，也就是正式上線時（temperature=0）會給的答案。這一步量到的錯誤率，最接近模型的真實表現。

結果會立刻存檔，萬一後面斷線也不會遺失。

In [ ]:
def save_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

g_texts, g_reasons, g_ntok, g_dt = generate(prompts, n=1, temperature=0.0)

records = []
for m, p, gold, t, fr in zip(meta, prompts, golds, g_texts, g_reasons):
    records.append({**m, "prompt": p, "gold": gold,
                    "greedy": {"text": t[0], "finish_reason": fr[0], **grade(t[0], gold)},
                    "samples": []})
save_jsonl(f"{OUT_DIR}/01a_probe_results.jsonl", records)

greedy_acc = sum(r["greedy"]["correct"] for r in records) / len(records)
print(f"Greedy 正確率 {greedy_acc:.1%}　|　{g_ntok} tokens / {g_dt:.0f}s = {g_ntok/g_dt:.0f} tok/s")

## 13. 抽樣：每題回答 4 次（分批跑、分批存檔）

調高 temperature 讓模型「冒險」，比較容易暴露它不確定的地方。

- 某題 greedy 答對、但抽樣有時答錯 → 模型在這題**不夠穩**，正好是 DPO 可以加強的地方
- 每次跑 100 題就存一次檔。100 題對 vLLM 來說已經足夠湊成大批次，分批不太影響速度

In [ ]:
CHUNK = 100
s_ntok, s_dt = 0, 0.0
for s in range(0, len(records), CHUNK):
    chunk = records[s:s+CHUNK]
    texts, reasons, ntok, dt = generate([r["prompt"] for r in chunk], n=N_SAMPLES, temperature=TEMPERATURE)
    s_ntok += ntok; s_dt += dt
    for r, ts, frs in zip(chunk, texts, reasons):
        r["samples"] = [{"text": t, "finish_reason": fr, **grade(t, r["gold"])} for t, fr in zip(ts, frs)]
    save_jsonl(f"{OUT_DIR}/01a_probe_results.jsonl", records)
    print(f"已完成 {min(s+CHUNK, len(records))}/{len(records)}　累計 {s_ntok/s_dt:.0f} tok/s")

## 14. 統計：各題型的錯誤率

| 欄位 | 意思 |
|---|---|
| `greedy_acc` | Greedy 正確率 |
| `sample_acc` | 所有抽樣回答中答對的比例 |
| `any_wrong` | 至少有一個回答（greedy 或抽樣）答錯的題目比例 → **這些題目都能做出「模型自己的錯」的配對** |
| `truncated` | 被 `max_tokens` 切斷的回答數，應該要是 0 |

In [ ]:
import pandas as pd

rows = []
for r in records:
    outs = [r["greedy"]] + r["samples"]
    rows.append({"category": r["category"],
                 "greedy_ok": r["greedy"]["correct"],
                 "sample_ok_rate": sum(o["correct"] for o in r["samples"]) / max(1, len(r["samples"])),
                 "any_wrong": any(not o["correct"] for o in outs),
                 "truncated": sum(o["finish_reason"] == "length" for o in outs)})
df = pd.DataFrame(rows)

table = df.groupby("category").agg(n=("greedy_ok", "size"), greedy_acc=("greedy_ok", "mean"),
                                   sample_acc=("sample_ok_rate", "mean"), any_wrong=("any_wrong", "mean"),
                                   truncated=("truncated", "sum"))
table.loc["ALL"] = [len(df), df.greedy_ok.mean(), df.sample_ok_rate.mean(), df.any_wrong.mean(), df.truncated.sum()]
display(table.style.format({"greedy_acc": "{:.1%}", "sample_acc": "{:.1%}", "any_wrong": "{:.1%}"}))

## 15. 錯誤類型分布

每一種錯誤標籤各出現幾次（greedy 和抽樣分開統計），以及推估：如果把整個 train（3,954 題）都跑一遍，大約能得到多少筆「模型自己的錯」的配對。

In [ ]:
err_greedy, err_sample = collections.Counter(), collections.Counter()
by_cat_err = collections.defaultdict(collections.Counter)
for r in records:
    for e in r["greedy"]["errors"]:
        err_greedy[e] += 1
    for o in r["samples"]:
        for e in o["errors"]:
            err_sample[e] += 1
            by_cat_err[r["category"]][e] += 1

print("Greedy 錯誤:", dict(err_greedy.most_common()))
print("抽樣錯誤:  ", dict(err_sample.most_common()))
print("\n各題型的抽樣錯誤:")
for c, cnt in sorted(by_cat_err.items()):
    print(f"  {c:10s}", dict(cnt.most_common()))

# 用每個題型的 any_wrong 比例，乘上 train 中該題型的總數
est_pairs = sum(df[df.category == c].any_wrong.mean() * len(idxs) for c, idxs in by_cat.items())
print(f"\n推估整個 train 能做出的 on-policy 配對（每題至少 1 筆）：約 {est_pairs:.0f} 筆")

## 16. 看幾個真實的錯誤

光看數字不夠，一定要親眼看錯誤長什麼樣子。這也是 01b 設計「規則造錯」的依據：**造出來的錯，要像模型真的會犯的錯。**

In [ ]:
shown, seen = collections.Counter(), set()
for r in records:
    for o in [r["greedy"]] + r["samples"]:
        if o["errors"] and shown[o["errors"][0]] < 2 and (r["id"], o["text"]) not in seen:
            shown[o["errors"][0]] += 1
            seen.add((r["id"], o["text"]))
            print(f"\n===== [{o['errors']}] {r['id']} ({r['category']}) =====")
            print("備註:", r["prompt"][-1]["content"].split("【操作員備註】")[-1])
            print("PRED:", o["text"][:600])
            print("GOLD:", r["gold"][:600])

## 17. 存檔

- `01a_probe_results.jsonl`：每題的完整結果（01b 會直接讀這個檔案）
- `01a_summary.json`：精簡摘要，**下載後傳給 Claude**

背景模式跑完後，這兩個檔案在 **Version → Output** 頁籤，會永久保存。01b 可以直接把這個 version 的 Output 加成輸入資料來讀取。

In [ ]:
summary = {
    "versions": VERSIONS, "backend": "vllm", "gpu": GPU_INFO, "model": MODEL_ID,
    "vllm_settings": {"dtype": "half", "max_model_len": 2048, "gpu_memory_utilization": 0.85,
                      "enforce_eager": True, "load_seconds": LOAD_SECONDS},
    "params": dict(N_TARGET=N_TARGET, MIN_PER_CATEGORY=MIN_PER_CATEGORY, N_SAMPLES=N_SAMPLES,
                   TEMPERATURE=TEMPERATURE, TOP_P=TOP_P, MAX_TOKENS=MAX_TOKENS, SEED=SEED),
    "plan": plan,
    "throughput_tok_s": {"greedy": round(g_ntok / g_dt), "sampling": round(s_ntok / s_dt)},
    "seconds": {"greedy": round(g_dt), "sampling": round(s_dt)},
    "table": table.reset_index().to_dict(orient="records"),
    "errors_greedy": dict(err_greedy), "errors_sample": dict(err_sample),
    "errors_by_category": {c: dict(v) for c, v in by_cat_err.items()},
    "est_onpolicy_pairs_full_train": round(est_pairs),
    "error_examples": [
        {"id": r["id"], "category": r["category"], "errors": o["errors"],
         "note": r["prompt"][-1]["content"].split("【操作員備註】")[-1], "pred": o["text"], "gold": r["gold"]}
        for r in records for o in [r["greedy"]] + r["samples"] if o["errors"]
    ][:40],
}
with open(f"{OUT_DIR}/01a_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=str)

print("✅ 全部完成。背景模式：到 Version → Output 下載 01a_summary.json 和 01a_probe_results.jsonl")